# Qwen2.5-7B VHE — final TEST evaluation
Run in a fresh Colab GPU runtime with the Drive containing the selected adapter.

In [ ]:
%pip install -q kagglehub datasets peft accelerate "bitsandbytes>=0.46.1" "transformers>=4.56,<5"

In [ ]:
from pathlib import Path
import json, sys
import kagglehub, torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from tqdm.auto import tqdm

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/Event-Extraction/Qwen2.5-7B-VHE-from-epoch8")
state = json.loads((root / "best_state.json").read_text())
epoch = state["best_epoch"]["baseline"]
assert epoch == 10, f"Expected the selected epoch 10, found {epoch}"
adapter = root / "best-baseline"
assert (adapter / "adapter_model.safetensors").is_file()
out = root / "test-evaluation"
out.mkdir(exist_ok=True)
assert not (out / "test-results.json").exists(), "Completed TEST results already exist on Drive"
assert torch.cuda.is_available(), "Select a GPU runtime"
kagglehub.login()
data = Path(kagglehub.dataset_download("md0612/vhe-adapted"))
sys.path.insert(0, kagglehub.dataset_download("md0612/metrics"))
from metrics.evaluator import Evaluator
rows = [json.loads(line) for line in (data / "test.jsonl").read_text().splitlines() if line.strip()]
assert len(rows) == 397
base = kagglehub.model_download("qwen-lm/qwen2.5/transformers/7b/1")
tok = AutoTokenizer.from_pretrained(adapter)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(base, quantization_config=quant, device_map="auto")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model = PeftModel.from_pretrained(model, adapter).eval()
model.config.use_cache = True
print(f"Loaded best epoch {epoch} | {len(rows)} TEST examples")


In [ ]:
predictions, golds = [], []
valid = 0
partial = out / "predictions.partial.jsonl"
with partial.open("w", encoding="utf-8") as stream, torch.inference_mode():
    for i, row in enumerate(tqdm(rows, desc="TEST baseline")):
        inputs = tok(row["text"], return_tensors="pt", add_special_tokens=True).to(model.device)
        ids = model.generate(**inputs, max_new_tokens=256, do_sample=False, num_beams=1,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id or tok.eos_token_id)
        pred = tok.decode(ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True,
                          clean_up_tokenization_spaces=False)
        gold = {"events": row["events"]}
        predictions.append(pred)
        golds.append(gold)
        try:
            obj = json.loads(pred)
            valid += int(isinstance(obj, dict) and isinstance(obj.get("events"), list))
        except json.JSONDecodeError:
            pass
        stream.write(json.dumps(dict(index=i, text=row["text"], gold=gold, prediction=pred), ensure_ascii=False) + "\n")
scores = Evaluator().evaluate_on_all_sample(predictions, golds)
result = dict(epoch=epoch, samples=len(rows), valid_json_rate=valid/len(rows), scores=scores)
payload = json.dumps(result, indent=2, ensure_ascii=False)
partial.replace(out / "test-predictions.jsonl")
tmp = out / "test-results.tmp"
tmp.write_text(payload, encoding="utf-8")
tmp.replace(out / "test-results.json")
print(payload)
